# LLM Cost Router

**The question this tool answers:** *"Am I overpaying by sending every request to my most expensive AI model?"*

Not every question needs your most powerful model. "What's the capital of France?" and "Analyze the trade-offs between microservices and a monolith" are wildly different in difficulty — but if you send both to the expensive frontier model, you massively overpay on the easy ones. Since **cost is the #1 concern for companies deploying AI**, this matters a lot.

A **cost router** looks at each question, judges how hard it is, and routes:
- **easy** questions → a cheap, small model
- **hard** questions → the expensive, powerful model

Same quality where it matters, a fraction of the cost. Think of a mailroom clerk: postcards go regular mail, only urgent contracts get the overnight courier.

### What's new here
Your other reliability projects *measure* quality. This one *takes an action to save money* — routing/triage logic plus a **cost-savings** headline number. It also connects back to the cost metric from the evaluation framework.

Run cells top to bottom with `Shift + Enter`.

## Step 1 — Setup

We use the Anthropic API for the (optional) real model calls and the LLM-judge classifier. Your key lives in Colab Secrets (🔑 panel), named `ANTHROPIC_API_KEY`.

In [ ]:
!pip install anthropic duckdb pandas -q
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
print("Ready.")

## Step 2 — The model catalog & pricing

Two tiers: a cheap small model and an expensive frontier model. Prices are **per 1,000,000 tokens** — placeholders here, so update them from the provider's pricing page for real numbers. The gap between cheap and expensive is what makes routing worthwhile.

In [ ]:
CHEAP_MODEL     = "claude-haiku-4-5-20251001"   # fast + cheap
EXPENSIVE_MODEL = "claude-sonnet-4-6"             # powerful + pricey

# USD per 1,000,000 tokens (input, output). UPDATE from the pricing page.
PRICING = {
    CHEAP_MODEL:     {"in": 0.25, "out": 1.25},
    EXPENSIVE_MODEL: {"in": 3.00, "out": 15.00},
}
print("Cheap:", CHEAP_MODEL)
print("Expensive:", EXPENSIVE_MODEL)

## Step 3 — The difficulty classifier (the router's brain)

The router needs to judge each question's difficulty. We show **two ways**:

1. **Rule-based** (fast, free, transparent) — looks at length, reasoning keywords, multi-part structure, and simple-lookup patterns.
2. **LLM-judge** (optional) — asks the cheap model itself "is this easy or hard?" More flexible, tiny cost.

We use the rule-based one by default because it's instant and free; the LLM-judge is there so you can compare. Both return "easy" or "hard".

In [ ]:
def estimate_difficulty_rules(question):
    """Cheap, transparent rules. Returns (label, reasons)."""
    q = question.lower()
    reasons, score = [], 0
    words = len(question.split())
    if words > 30:
        score += 2; reasons.append("long question")
    elif words > 15:
        score += 1; reasons.append("medium length")

    hard_kw = ["analyze","compare","explain why","summarize","evaluate","design",
               "debug","trade-off","tradeoff","implications","step by step","prove",
               "derive","critique","reason"]
    if any(k in q for k in hard_kw):
        score += 2; reasons.append("reasoning/analysis task")

    if q.count("?") > 1 or (" and " in q and words > 12):
        score += 1; reasons.append("multi-part")

    easy_kw = ["what is","who is","when did","define","capital of","how many",
               "convert","translate","spell"]
    if any(q.startswith(k) or k in q for k in easy_kw) and words <= 15:
        score -= 1; reasons.append("simple lookup")

    label = "hard" if score >= 2 else "easy"
    return label, (reasons or ["default"])


def estimate_difficulty_llm(question):
    """Ask the cheap model to classify. Returns 'easy' or 'hard'."""
    resp = client.messages.create(
        model=CHEAP_MODEL, max_tokens=5,
        messages=[{"role":"user","content":
            f"Classify this question as exactly one word, 'easy' or 'hard', "
            f"based on whether it needs deep reasoning. Question: {question}"}],
    )
    out = resp.content[0].text.strip().lower()
    return "hard" if "hard" in out else "easy"

print("Classifiers defined.")

## Step 4 — The router and cost math

`route` picks a tier from the difficulty. `token_cost` turns a call into dollars. We assume a fixed token count per call so the comparison isolates the *routing* effect (not variation in answer length).

In [ ]:
def route(question):
    label, reasons = estimate_difficulty_rules(question)
    tier_model = EXPENSIVE_MODEL if label == "hard" else CHEAP_MODEL
    return {"model": tier_model, "difficulty": label, "reasons": reasons}

def token_cost(model, in_tokens, out_tokens):
    p = PRICING[model]
    return (in_tokens/1_000_000)*p["in"] + (out_tokens/1_000_000)*p["out"]

print("Router defined.")

## Step 5 — A realistic mixed workload

A batch of 12 questions: some trivial lookups, some genuinely hard reasoning tasks — the kind of mix a real AI product receives.

In [ ]:
workload = [
    "What is the capital of France?",
    "Who is the CEO of Microsoft?",
    "Convert 100 USD to EUR",
    "Define photosynthesis",
    "How many days are in a leap year?",
    "Translate 'good morning' to Spanish",
    "Analyze the trade-offs between microservices and a monolith for a startup",
    "Explain why transformer models scale better than RNNs, step by step",
    "Compare the tax implications of an LLC versus an S-corp and recommend one",
    "Summarize this contract and flag clauses that expose us to liability risk",
    "Debug why my recursive function causes a stack overflow and suggest a fix",
    "Evaluate whether we should migrate our database to a distributed system",
]
print(f"{len(workload)} questions in the workload.")

## Step 6 — Route the batch and measure savings 💰

For each question: classify, pick a tier, compute its routed cost, and compare against the **baseline** (sending everything to the expensive model). We assume ~400 input / ~200 output tokens per call.

In [ ]:
AVG_IN, AVG_OUT = 400, 200

rows, routed_total, baseline_total = [], 0.0, 0.0
for q in workload:
    r = route(q)
    c_routed = token_cost(r["model"], AVG_IN, AVG_OUT)
    c_base   = token_cost(EXPENSIVE_MODEL, AVG_IN, AVG_OUT)
    routed_total += c_routed
    baseline_total += c_base
    rows.append({"question": q, "difficulty": r["difficulty"],
                 "model": r["model"], "reasons": ", ".join(r["reasons"]),
                 "routed_cost": c_routed, "baseline_cost": c_base})

savings = baseline_total - routed_total
pct = savings / baseline_total * 100

n_cheap = sum(1 for r in rows if r["model"] == CHEAP_MODEL)
n_exp   = len(rows) - n_cheap

for r in rows:
    tier = "CHEAP" if r["model"] == CHEAP_MODEL else "EXPENSIVE"
    print(f"  [{r['difficulty']:4}] {tier:9} | {r['question'][:52]}")

print()
print(f"Routed to cheap:     {n_cheap}/{len(rows)} questions")
print(f"Routed to expensive: {n_exp}/{len(rows)} questions")
print(f"Cost WITH routing:   ${routed_total:.5f}")
print(f"Cost WITHOUT (all expensive): ${baseline_total:.5f}")
print(f"SAVINGS: ${savings:.5f}  ({pct:.0f}% cheaper)")

## Step 7 — (Optional) Prove quality holds on easy questions

Saving money only matters if the cheap model still answers the easy questions correctly. This cell actually calls both models on one easy question so you can eyeball that the cheap answer is just as good — for easy questions, it usually is. (Costs a fraction of a cent.)

In [ ]:
easy_q = "What is the capital of France?"

cheap_ans = client.messages.create(model=CHEAP_MODEL, max_tokens=50,
    messages=[{"role":"user","content":easy_q}]).content[0].text.strip()
exp_ans = client.messages.create(model=EXPENSIVE_MODEL, max_tokens=50,
    messages=[{"role":"user","content":easy_q}]).content[0].text.strip()

print("Question:", easy_q)
print("\nCHEAP model:    ", cheap_ans)
print("EXPENSIVE model:", exp_ans)
print("\n-> Same correct answer, cheaper model. That is the routing thesis.")

## Step 8 — Save results for the dashboard

In [ ]:
import duckdb
from datetime import datetime

con = duckdb.connect("router_results.db")
con.execute("CREATE TABLE IF NOT EXISTS routed (question VARCHAR, difficulty VARCHAR, model VARCHAR, reasons VARCHAR, routed_cost DOUBLE, baseline_cost DOUBLE)")
con.execute("CREATE TABLE IF NOT EXISTS totals (checked_at TIMESTAMP, n INTEGER, n_cheap INTEGER, n_expensive INTEGER, routed_cost DOUBLE, baseline_cost DOUBLE, savings DOUBLE, savings_pct DOUBLE)")
con.execute("DELETE FROM routed"); con.execute("DELETE FROM totals")

for r in rows:
    con.execute("INSERT INTO routed VALUES (?,?,?,?,?,?)",
        [r["question"], r["difficulty"], r["model"], r["reasons"], r["routed_cost"], r["baseline_cost"]])
con.execute("INSERT INTO totals VALUES (?,?,?,?,?,?,?,?)",
    [datetime.now(), len(rows), n_cheap, n_exp, routed_total, baseline_total, savings, pct])
con.close()
print("Saved to router_results.db")
print("Download it (Files panel) and drop it into the dashboard's data/ folder.")

## ✅ Done — what you built

An LLM cost router that:
- **classifies** each question's difficulty (cheap rules, or an LLM-judge),
- **routes** easy questions to a cheap model and hard ones to an expensive model,
- **measures the cost savings** versus sending everything to the expensive model,
- and saves results for a dashboard.

**The takeaway to explain in interviews:** the biggest barrier to deploying AI at scale is cost. A router cuts spend substantially with no quality loss on easy traffic — and unlike a pure monitoring tool, it *acts* to save money. This rounds out your reliability portfolio with an efficiency/cost angle.

**To make it yours:** plug in your own workload, switch to the LLM-judge classifier, tune the difficulty rules, or add a third "medium" tier.

**Next:** download `router_results.db` and open the Streamlit dashboard.